# Verificación de ITCHI v0.2 contra declaratorias: métricas, figuras e interpretación

Este notebook reproduce las figuras y tablas de la presentación de avances. Explica cada métrica y cómo leer cada resultado.

**Entradas.** Son las salidas de los scripts, que hay que correr antes:
- `scripts/declaratorias_6h.py` → `declaratorias_ct_eventos.csv`
- `scripts/negativos_w06.py` → `pares_municipio_ct_w06.csv.gz`
- `scripts/percentiles_6h.py` → `mswep_v3_{6,24,72}h_percentiles_may_nov.nc`

**Salidas.** Figuras PNG y tablas CSV en `outputs/figures/verificacion_v02/`.

Contenido:
1. Métricas de verificación: qué miden y por qué usamos PSS y AUC.
2. Climatología por duración.
3. Con qué lluvia se emiten las declaratorias.
4. Probabilidad de declaratoria por celda P6 × A72.
5. Persistencia (R24) y antecedente (A72) dentro de P95–P99.
6. Comparación de esquemas de niveles.
7. Curva ROC.
8. La regla continua de lluvia de la v0.2.
9. El perfil de viento de v0.1 y v0.2.
10. ¿Aporta información el viento fuera de R34? (componente indirecto)

## 0. Configuración

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from itchi.config import get_project_root
from itchi.parametric_radii import compute_w06_wind_hazard_with_tail
from itchi.precipitation import RAMP_R4_NODES, PRECIP_MATRIX_V02, compute_precipitation_hazard_ramp
from itchi.wind import compute_wind_hazard_from_profile
from itchi.wind_profiles import KNOT_TO_MS, Willoughby2006Profile

ROOT = get_project_root()
DIAG = ROOT / "outputs" / "diagnostics" / "declaratorias_6h"
FIG = ROOT / "outputs" / "figures" / "verificacion_v02"
FIG.mkdir(parents=True, exist_ok=True)

pares = pd.read_csv(DIAG / "pares_municipio_ct_w06.csv.gz", dtype={"CVEGEO": str}, parse_dates=["t_pico"])
pares = pares[pares["nivel_pico"] >= 0].copy()          # pares con lluvia válida en el pico
pares["year"] = pares["t_pico"].dt.year

# Niveles recalculados desde las clases, para no depender de la versión de la regla con que se
# generó el CSV: rampa de 6 h (R4) y v0.2 (P95–P99 -> máximo si R24 >= P95 y A72 >= P75)
_fila = pares["nivel_pico"].astype(int).to_numpy()
pares["nivel_base"] = np.array([0, 1, 2, 3, 4, 4])[_fila]
_esc = ((_fila == 3) & (pares["nivel_R24_en_pico6"].fillna(-1).to_numpy() >= 3)
        & (pares["nivel_A72_en_pico6"].fillna(-1).to_numpy() >= 1))
pares["nivel_v02"] = np.where(_esc, 4, pares["nivel_base"])
eventos = pd.read_csv(DIAG / "declaratorias_ct_eventos.csv", dtype={"CVEGEO": str})

D = pares["D"].to_numpy(bool)                             # 1 = hubo declaratoria asociada al ciclón
D_DES = pares["D_desastre"].to_numpy(bool)
CLASES = ["<P75", "P75–P90", "P90–P95", "P95–P99", "P99–P99.9", "≥P99.9"]
COL = {"base": "#6B7280", "v0.2": "#2A6FC9", "alt": "#D9822B"}
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
print(f"pares: {len(pares):,} | positivos: {D.sum():,} ({100 * D.mean():.1f} %) | desastres: {D_DES.sum():,}")

## 1. Métricas de verificación

Cada esquema asigna un **nivel** (0 sin peligro, 1 bajo, 2 medio, 3 alto, 4 máximo) a cada par municipio–ciclón. Para un umbral de alerta "nivel ≥ k", cada par cae en una casilla de la tabla de contingencia:

| | Hubo declaratoria (D = 1) | No hubo (D = 0) |
|---|---|---|
| **Alerta** (nivel ≥ k) | a: aciertos | b: falsas alarmas |
| **Sin alerta** | c: fallos | d: negativos correctos |

| Métrica | Fórmula | Pregunta que responde | Rango |
|---|---|---|---|
| POD | a / (a + c) | De los casos con declaratoria, ¿qué fracción tuvo alerta? | 0–1, mejor alto |
| POFD | b / (b + d) | De los casos sin declaratoria, ¿qué fracción tuvo alerta? | 0–1, mejor bajo |
| FAR | b / (a + b) | De las alertas, ¿qué fracción no tuvo declaratoria? | 0–1, mejor bajo |
| CSI | a / (a + b + c) | Aciertos entre todo lo que fue evento o alerta | 0–1 |
| Sesgo | (a + b) / (a + c) | ¿Cuántas alertas por cada evento real? | 1 = sin sesgo |
| **PSS** | POD − POFD | ¿Cuánto separa el umbral los dos grupos? | −1 a 1; 0 = azar |
| **AUC** | área bajo la curva ROC | Probabilidad de que un caso con declaratoria tenga más nivel que uno sin ella | 0.5 = azar, 1 = perfecto |

**Por qué PSS y AUC, y no FAR ni CSI.** Solo el 2 % de los pares tiene declaratoria. FAR y CSI dependen de esa tasa base: aunque el índice fuera bueno, la mayoría de las alertas caerían en municipios sin declaratoria, porque son el 98 %. Además, las declaratorias reflejan también la exposición y la gestión, y tienen subregistro.

POD y POFD se calculan cada una dentro de su propio grupo y no dependen de la tasa base. Su diferencia, PSS (Peirce o Hanssen–Kuipers), es la distancia vertical de cada punto de la curva ROC a la diagonal del azar. El umbral con el PSS más alto es el que mejor discrimina. Referencia: Jolliffe y Stephenson (2012), *Forecast Verification*, 2.ª ed., Wiley.

In [ ]:
def contingency(alert: np.ndarray, obs: np.ndarray) -> dict:
    a = int(np.sum(alert & obs)); b = int(np.sum(alert & ~obs))
    c = int(np.sum(~alert & obs)); d = int(np.sum(~alert & ~obs))
    pod, pofd = a / (a + c), b / (b + d)
    return {"a": a, "b": b, "c": c, "d": d, "POD": pod, "POFD": pofd, "PSS": pod - pofd,
            "FAR": b / (a + b) if a + b else np.nan, "CSI": a / (a + b + c), "sesgo": (a + b) / (a + c)}


def auc(score: np.ndarray, obs: np.ndarray) -> float:
    # AUC por rangos (Mann–Whitney); los empates cuentan la mitad
    r = pd.Series(score).rank().to_numpy()
    n1, n0 = obs.sum(), (~obs).sum()
    return (r[obs].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def auc_por_anio(score: np.ndarray, obs: np.ndarray = D) -> float:
    # Media del AUC de cada año: evita mezclar años con tasas base muy distintas (0.2 %–9 %)
    df = pd.DataFrame({"s": score, "d": obs, "y": pares["year"].to_numpy()})
    return float(np.mean([auc(g["s"].to_numpy(), g["d"].to_numpy()) for _, g in df.groupby("y") if 0 < g["d"].sum() < len(g)]))


# Ejemplo: rampa de 6 h, alerta si nivel >= 3 (P6 >= P95)
ej = contingency(pares["nivel_base"].to_numpy() >= 3, D)
pd.Series(ej).round(3)

**Cómo leer el ejemplo.** Con la alerta en nivel ≥ 3 (P6 ≥ P95):
- POD ≈ 0.71: 7 de cada 10 pares con declaratoria tuvieron lluvia ≥ P95 en su pico.
- POFD ≈ 0.16: 16 % de los municipios sin declaratoria también la tuvieron.
- PSS ≈ 0.55: el umbral separa bien los dos grupos.
- FAR ≈ 0.92 y sesgo ≈ 8.7: hay muchas más alertas que declaratorias, por la tasa base y el subregistro.

### 1.1 Las métricas en gráficas

Tres figuras que conviene mostrar juntas:

1. **La tabla de contingencia** de un umbral, con sus cuatro casillas: es de donde salen todas las métricas.
2. **Cada métrica en función del umbral de alerta (nivel ≥ k)**, para la rampa de 6 h y para la v0.2. Así se ve el compromiso: al subir el umbral baja la detección (POD), pero también la falsa detección (POFD).
3. **AUC y tasa base por año.** Se ve por qué la evaluación se hace con métricas que no dependen de la tasa base.

**Por qué importa cada una:**
- **POD** responde si el índice "ve" los eventos. Un índice que pierde muchos desastres no sirve para alertar.
- **POFD** responde cuánto alarma en vano entre los casos sin evento. Con un universo de ~380 mil pares, unos puntos de POFD son muchos municipios.
- **PSS** combina las dos en un número que no depende de la tasa base. El umbral donde es máximo es el que mejor separa los grupos. Sirve para comparar esquemas y para elegir cortes.
- **FAR y CSI** dependen de la tasa base (2 %). Se reportan para transparencia, pero no se usan para decidir. Una FAR alta aquí no significa que el índice sea malo.
- **Sesgo de frecuencia** dice cuántas alertas se emiten por cada evento. Mayor que 1 es sobrealertar. Aquí es esperable por el subregistro de declaratorias.
- **AUC** mide el ordenamiento completo, sin elegir un umbral. Es la medida más general de la capacidad del índice para separar casos.

In [ ]:
def plot_contingencia(ax, alerta, obs, titulo):
    s = contingency(alerta, obs)
    casillas = [[("aciertos", s["a"], "#B9D3F2"), ("falsas alarmas", s["b"], "#F3D2B3")],
                [("fallos", s["c"], "#F2B8A6"), ("negativos correctos", s["d"], "#E5E3DC")]]
    for i in range(2):
        for j in range(2):
            nombre, n, color = casillas[i][j]
            ax.add_patch(plt.Rectangle((j, 1 - i), 1, 1, color=color, ec="white", lw=3))
            ax.text(j + 0.5, 1.62 - i, nombre, ha="center", va="center", fontsize=10, color="#14213D")
            ax.text(j + 0.5, 1.4 - i, f"{n:,}", ha="center", va="center", fontsize=14, weight="bold", color="#14213D")
    ax.set(xlim=(0, 2), ylim=(0, 2), xticks=[0.5, 1.5], yticks=[0.5, 1.5], title=titulo)
    ax.set_xticklabels(["con declaratoria", "sin declaratoria"]); ax.set_yticklabels(["sin alerta", "alerta"])
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(1, -0.32, f"POD = {s['POD']:.2f}   POFD = {s['POFD']:.2f}   PSS = {s['PSS']:.2f}   FAR = {s['FAR']:.2f}",
            ha="center", fontsize=10, transform=ax.transData)


fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
plot_contingencia(axes[0], pares["nivel_base"].to_numpy() >= 4, D, "Rampa de 6 h, alerta si nivel ≥ 4")
plot_contingencia(axes[1], pares["nivel_v02"].to_numpy() >= 4, D, "v0.2, alerta si nivel ≥ 4")
fig.tight_layout()
fig.savefig(FIG / "contingencia_nivel4.png", dpi=200, bbox_inches="tight")

**Cómo leer.** La v0.2 mueve casos de "fallos" a "aciertos" (sube POD) y, a cambio, algunos de "negativos correctos" a "falsas alarmas" (sube POFD). Como el primer cambio es proporcionalmente mayor, el PSS sube.

In [ ]:
ESQ_MET = {"Rampa de 6 h (R4)": (pares["nivel_base"].to_numpy(), dict(color=COL["base"], ls="--", marker="o")),
           "v0.2": (pares["nivel_v02"].to_numpy(), dict(color=COL["v0.2"], ls="-", marker="s"))}
filas = []
for nombre, (lv, _) in ESQ_MET.items():
    for k in range(1, 5):
        filas.append({"esquema": nombre, "nivel ≥": k, **{m: v for m, v in contingency(lv >= k, D).items() if m not in "abcd"}})
MET = pd.DataFrame(filas)
MET.to_csv(FIG / "metricas_por_umbral.csv", index=False, float_format="%.3f")

fig, axes = plt.subplots(2, 3, figsize=(12, 6.6), sharex=True)
titulos = {"POD": "POD: detección (↑ mejor)", "POFD": "POFD: falsa detección (↓ mejor)",
           "PSS": "PSS = POD − POFD (↑ mejor)", "FAR": "FAR (depende de la tasa base)",
           "CSI": "CSI (depende de la tasa base)", "sesgo": "Sesgo de frecuencia (1 = sin sesgo)"}
for ax, m in zip(axes.ravel(), titulos):
    for nombre, (_, est) in ESQ_MET.items():
        v = MET[MET["esquema"] == nombre]
        ax.plot(v["nivel ≥"], v[m], lw=2, label=nombre, **est)
    if m == "sesgo":
        ax.set_yscale("log"); ax.axhline(1, color="0.6", lw=0.8, ls=":")
    ax.set_title(titulos[m], fontsize=11)
    ax.set_xticks(range(1, 5))
for ax in axes[1]:
    ax.set_xlabel("alerta si nivel ≥")
axes[0, 0].legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "metricas_por_umbral.png", dpi=200, bbox_inches="tight")
MET.round(3)

**Cómo leer.**
- POD y POFD bajan juntas al subir el umbral; PSS resume el balance.
- La v0.2 coincide con la rampa en los niveles 1–3 y solo cambia el nivel 4: más POD y algo más de POFD.
- FAR y CSI son casi planas y desfavorables en todos los umbrales porque reflejan la tasa base, no la calidad del índice.
- El sesgo mayor que 1 indica sobrealerta. Con el subregistro de declaratorias, es esperable y en parte deseable para alerta temprana.

In [ ]:
def auc_seguro(s, y):
    return auc(s, y) if 0 < y.sum() < len(y) else np.nan


anual = pares.groupby("year").apply(lambda g: pd.Series({
    "tasa base (%)": 100 * g["D"].mean(),
    "AUC rampa de 6 h": auc_seguro(g["nivel_base"].to_numpy(), g["D"].to_numpy(bool)),
    "AUC v0.2": auc_seguro(g["nivel_v02"].to_numpy(), g["D"].to_numpy(bool)),
    "pares": len(g)}), include_groups=False)
anual.to_csv(FIG / "auc_por_anio.csv", float_format="%.3f")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
x = anual.index.to_numpy()
axes[0].bar(x - 0.2, anual["AUC rampa de 6 h"], 0.4, color=COL["base"], hatch="//", edgecolor="white", label="rampa de 6 h")
axes[0].bar(x + 0.2, anual["AUC v0.2"], 0.4, color=COL["v0.2"], edgecolor="white", label="v0.2")
axes[0].axhline(0.5, color="0.6", ls=":", lw=1)
axes[0].set(ylim=(0.4, 1.0), ylabel="AUC", title="AUC por año (0.5 = azar)")
axes[0].legend(frameon=False, ncol=2, fontsize=9)
axes[1].bar(x, anual["tasa base (%)"], 0.6, color="#B9C2CF")
axes[1].set(ylabel="% de pares con declaratoria", title="Tasa base por año")
fig.tight_layout()
fig.savefig(FIG / "auc_y_tasa_base_por_anio.png", dpi=200, bbox_inches="tight")
anual.round(3)

**Cómo leer.**
- La tasa base cambia mucho entre años, de ~0.2 % a ~9 %. Depende de qué tan activa fue la temporada en México y de la gestión de declaratorias.
- Aun así, el AUC se mantiene alto y estable casi todos los años: la capacidad de separar casos no depende de la tasa base.
- Por eso el AUC se resume como promedio por año, y por eso los intervalos de confianza deben calcularse remuestreando ciclones o años, no pares sueltos.
- Un año con AUC cercano o menor a 0.5 (en estos datos, 2000) indica un problema de datos más que del índice. Lo más probable es la asociación declaratoria–ciclón por fechas, y hay que revisarlo antes de publicar.

## 2. Climatología por duración (medianas sobre México)

In [ ]:
import xarray as xr

caja = dict(lat=slice(14, 33.5), lon=slice(-118.5, -86))
filas = {}
for d in (6, 24, 72):
    with xr.open_dataset(ROOT / "data" / "processed" / f"mswep_v3_{d}h_percentiles_may_nov.nc") as ds:
        q = ds["p_wet_raw"].sel(**caja).median(("lat", "lon")).values
        filas[f"{d} h"] = dict(zip([f"P{v * 100:g}" for v in ds["quantile"].values], np.round(q, 1)))
        filas[f"{d} h"]["frecuencia húmeda"] = round(float(ds["wet_freq"].sel(**caja).median()), 2)
clim = pd.DataFrame(filas).T
clim.to_csv(FIG / "climatologia_medianas.csv")
clim

**Cómo leer la tabla.** Son los umbrales locales, medianos sobre México, en mm por ventana húmeda.
- Los acumulados crecen menos que proporcionalmente con la duración (72 h ≈ 1.4–1.6 × 24 h), como corresponde a lluvia intermitente.
- En 72 h, P99 y P99.9 no son robustos: las ventanas se traslapan y la muestra efectiva es n/12. La regla de la v0.2 usa solo P75–P95 de 72 h.

## 3. Con qué lluvia se emiten las declaratorias (solo positivos)

In [ ]:
ev = eventos[eventos["P_evento"].notna()].copy()
lv = ev["nivel_pico"]
tabla_tipos = ev.groupby("Tipo Declaratoria").apply(lambda g: pd.Series({
    "n": len(g),
    "% < 1 mm": 100 * (g["P_evento"] < 1).mean(),
    "% < P75": 100 * (g["nivel_pico"] == 0).mean(),
    "% >= P95": 100 * (g["nivel_pico"] >= 3).mean(),
    "% >= P99": 100 * (g["nivel_pico"] >= 4).mean(),
}), include_groups=False).round(1)
tabla_tipos.to_csv(FIG / "declaratorias_por_tipo.csv")

frac = pd.crosstab(ev["Tipo Declaratoria"], ev["nivel_pico"].astype(int), normalize="index").reindex(columns=range(6), fill_value=0)
siat = ["#E5E3DC", "#9CC3EE", "#A8D08D", "#F5C451", "#EE9A4D", "#E06666"]
fig, ax = plt.subplots(figsize=(9, 2.8))
left = np.zeros(len(frac))
for k in range(6):
    ax.barh(frac.index, frac[k], left=left, color=siat[k], edgecolor="white", label=CLASES[k])
    left += frac[k].to_numpy()
ax.set(xlim=(0, 1), xlabel="fracción de declaratorias")
ax.invert_yaxis()
ax.legend(ncol=6, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.3))
fig.savefig(FIG / "declaratorias_clase_por_tipo.png", dpi=200, bbox_inches="tight")
tabla_tipos

**Cómo leer.**
- P75 cubre a casi todos los desastres (< P75: ~4 %): es una buena alerta temprana.
- Más del 80 % de los desastres rebasa P95. El nivel máximo tiene que separar lo excepcional dentro de esa franja.
- Alrededor del 18 % de las emergencias tuvo menos de 1 mm. Son preventivas o por otro fenómeno, y ningún umbral de lluvia las captura.
- Esta tabla solo tiene positivos: dice con qué lluvia se declaró, no cuántas veces la misma lluvia ocurrió sin declaratoria. Por eso hacen falta los negativos de las secciones siguientes.

## 4. Probabilidad de declaratoria por celda P6 × A72

In [ ]:
grupoA = pares["grupo_A72_pico"].fillna("<P75")
tab = pares.assign(gA=grupoA).groupby(["nivel_pico", "gA"])["D"].agg(["size", "mean"])
P = (100 * tab["mean"]).unstack()[["<P75", "P75-P95", ">=P95"]]
N = tab["size"].unstack()[["<P75", "P75-P95", ">=P95"]]
P.index = CLASES; N.index = CLASES

fig, ax = plt.subplots(figsize=(7, 5.4))
im = ax.imshow(P.to_numpy(), cmap="Blues", norm=plt.matplotlib.colors.LogNorm(0.3, 45))
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        v = P.iat[i, j]
        ax.text(j, i, f"{v:.1f} %\nn={N.iat[i, j]:,}", ha="center", va="center", fontsize=9,
                color="white" if v > 12 else "#14213D")
ax.set_xticks(range(3), ["A72\n< P75", "A72\nP75–P95", "A72\n≥ P95"]); ax.set_yticks(range(6), CLASES)
ax.set(xlabel="antecedente de 72 h", ylabel="clase de P6 en el pico")
cb = fig.colorbar(im, ax=ax, label="P(declaratoria) %", ticks=[0.5, 1, 2, 5, 10, 20, 40])
cb.ax.set_yticklabels(["0.5", "1", "2", "5", "10", "20", "40"])
fig.savefig(FIG / "probabilidad_por_celda.png", dpi=200, bbox_inches="tight")
P.round(1)

**Cómo leer.** Cada celda es la frecuencia observada de declaratoria entre los pares que caen en ella; la tasa base global es de 2 %.
- **Hacia abajo** la probabilidad crece con la lluvia de 6 h: de 0.5 % a 11 % sin antecedente.
- **Hacia la derecha**, el antecedente casi no cambia nada debajo de P90 y triplica la probabilidad a partir de P95.

Es una interacción, no una suma: el suelo húmedo amplifica la lluvia intensa, pero no convierte la moderada en peligrosa. Por eso la escalada de la v0.2 solo actúa a partir de Q90 de P6 y es máxima en Q95.

## 5. Persistencia y antecedente dentro de P95–P99

In [ ]:
def grupo(c):  # clase 0..5 -> 0: <P75, 1: P75–P95, 2: >=P95
    return np.select([c >= 3, c >= 1], [2, 1], 0)

sub = pares[pares["nivel_pico"] == 3]
t = sub.assign(gR24=np.where(sub["nivel_R24_en_pico6"] >= 3, "R24 ≥ P95", "R24 < P95"),
               gA=grupo(sub["nivel_A72_en_pico6"].fillna(-1)))
pers = (100 * t.groupby(["gR24", "gA"])["D"].mean()).unstack().rename(columns={0: "A72 < P75", 1: "A72 P75–P95", 2: "A72 ≥ P95"})
pers.to_csv(FIG / "persistencia_P95_P99.csv")
pers.round(1)

**Cómo leer.** Dentro de la fila P95–P99:
- Sin persistencia (R24 < P95), el antecedente no aporta (~2 %).
- Con lluvia sostenida, el antecedente multiplica la probabilidad: de 4 % a 10 %.

De aquí sale la regla de la v0.2: P95–P99 pasa a máximo si R24 ≥ P95 y A72 ≥ P75.

## 6. Comparación de esquemas de niveles

In [ ]:
base = np.array([0, 1, 2, 3, 4, 4])[pares["nivel_pico"].astype(int).to_numpy()]
c24 = pares["nivel_R24_en_pico6"].fillna(-1).to_numpy()
c72 = pares["nivel_R72_en_pico6"].fillna(-1).to_numpy()
a72 = pares["nivel_A72_en_pico6"].fillna(-1).to_numpy()
fila = pares["nivel_pico"].astype(int).to_numpy()


def escalada(cond):
    lv = base.copy(); lv[(fila == 3) & cond] = 4; return lv


ESQUEMAS = {
    "Rampa de 6 h (R4)": base,
    "Máximo de clases 6/24/72 h": np.array([0, 1, 2, 3, 4, 4])[np.maximum.reduce([fila, np.maximum(c24, 0), np.minimum(np.maximum(c72, 0), 3)]).astype(int)],
    "Escalada si A72 ≥ P95": escalada(a72 >= 3),
    "Escalada si R24 ≥ P99 o A72 ≥ P95": escalada((c24 >= 4) | (a72 >= 3)),
    "v0.2: R24 ≥ P95 y A72 ≥ P75": escalada((c24 >= 3) & (a72 >= 1)),
}
filas = []
for nombre, lv in ESQUEMAS.items():
    r = {"esquema": nombre, "AUC/año": auc_por_anio(lv)}
    for k in range(1, 5):
        s = contingency(lv >= k, D); r[f"PSS ≥ {k}"] = s["PSS"]
    s4 = contingency(lv >= 4, D); r["POD ≥ 4"], r["POFD ≥ 4"] = s4["POD"], s4["POFD"]
    keep = ~D | D_DES
    r["PSS ≥ 4 desastres"] = contingency((lv >= 4)[keep], D_DES[keep])["PSS"]
    filas.append(r)
ESQ = pd.DataFrame(filas).set_index("esquema")
ESQ.to_csv(FIG / "esquemas_pss.csv", float_format="%.3f")

fig, ax = plt.subplots(figsize=(9, 3.8))
w = 0.16
for j, (nombre, row) in enumerate(ESQ.iterrows()):
    ax.bar(np.arange(1, 5) + (j - 2) * w, row[[f"PSS ≥ {k}" for k in range(1, 5)]], w, label=nombre,
           color=["#6B7280", "#B9C2CF", "#E8A25A", "#C2571A", "#2A6FC9"][j], edgecolor="white")
ax.set(xticks=range(1, 5), xticklabels=[f"nivel ≥ {k}" for k in range(1, 5)], ylabel="PSS", ylim=(0.3, 0.6))
ax.legend(fontsize=8, frameon=False, ncol=2)
fig.savefig(FIG / "esquemas_pss.png", dpi=200, bbox_inches="tight")
ESQ.round(3)

**Cómo leer.**
- El AUC por año es casi igual en todos (~0.86): los cinco ordenan los pares casi igual. La diferencia está en dónde pone cada uno los cortes.
- Las escaladas dejan los niveles 1–3 idénticos a la rampa y solo cambian el máximo.
- El "máximo de clases" empeora los niveles bajos: sube de nivel donde el antecedente no aporta.
- En la v0.2, el máximo detecta 60 % de los pares con declaratoria (antes 47 %), a cambio de activarse en 8 % de los pares sin ella (antes 5 %).

La regla se eligió con estos mismos datos. Falta validación cruzada por año o ciclón, y bootstrap por ciclón para los intervalos.

## 7. Curva ROC por nivel

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5))
for nombre, estilo in (("Rampa de 6 h (R4)", dict(color=COL["base"], ls="--", marker="o")),
                       ("v0.2: R24 ≥ P95 y A72 ≥ P75", dict(color=COL["v0.2"], ls="-", marker="s"))):
    lv = ESQUEMAS[nombre]
    pts = [contingency(lv >= k, D) for k in (4, 3, 2, 1)]
    ax.plot([0] + [p["POFD"] for p in pts] + [1], [0] + [p["POD"] for p in pts] + [1], lw=2, label=nombre, **estilo)
    for k, p in zip((4, 3, 2, 1), pts):
        ax.annotate(f"≥{k}", (p["POFD"], p["POD"]), textcoords="offset points", xytext=(6, -12), fontsize=9)
ax.plot([0, 1], [0, 1], ":", color="0.6", label="azar (PSS = 0)")
ax.set(xlim=(0, 0.6), ylim=(0.3, 1.0), xlabel="POFD (falsa detección)", ylabel="POD (detección)")
ax.legend(frameon=False, fontsize=9, loc="lower right")
fig.savefig(FIG / "roc_niveles.png", dpi=200, bbox_inches="tight")

**Cómo leer.** Cada punto es un umbral de alerta. Más arriba y más a la izquierda es mejor.
- PSS es la distancia vertical de cada punto a la diagonal.
- Los niveles 1–3 coinciden en los dos esquemas.
- En el nivel 4 la v0.2 sube: +0.13 de POD por +0.03 de POFD.

## 8. La regla continua de lluvia (como la usa ITCHI)

In [ ]:
q6 = {"Q75": 9.8, "Q90": 20.1, "Q95": 29.3, "Q99": 52.5}       # medianas de México, 6 h
p = np.linspace(0, 80, 801)
nodos = PRECIP_MATRIX_V02["p6_nodes"]
r4 = compute_precipitation_hazard_ramp(p, q6, RAMP_R4_NODES)
esc = compute_precipitation_hazard_ramp(p, q6, dict(zip(nodos, PRECIP_MATRIX_V02["escalated"][1:])))
fig, ax = plt.subplots(figsize=(7, 4))
for y0 in (0.2, 0.6):
    ax.axhspan(y0, y0 + 0.2, color="0.95", lw=0)
for k, v in q6.items():
    ax.axvline(v, color="0.75", ls=":", lw=1); ax.text(v, 1.04, k, ha="center", fontsize=9)
ax.plot(p, r4, "--", color=COL["base"], lw=2, label="R4 (sin escalada)")
for f in (0.5,):
    ax.plot(p, (1 - f) * r4 + f * esc, color=COL["v0.2"], lw=1.2, alpha=0.6, label=f"escalada parcial (f = {f})")
ax.plot(p, esc, color=COL["v0.2"], lw=2.5, label="escalada total (f = 1)")
ax.set(xlabel="P6 (mm en 6 h)", ylabel="H_P", ylim=(0, 1.1), xlim=(0, 80))
ax.legend(frameon=False, fontsize=9)
fig.savefig(FIG / "regla_lluvia_continua.png", dpi=200, bbox_inches="tight")

**Cómo leer.** $H_P = (1-f)\,R4 + f\,\mathrm{Esc}$, con $f = f_{R24} \cdot f_{A72}$.
- $f_{R24}$ va de 0 en Q90₂₄ a 1 en Q95₂₄.
- $f_{A72}$ va de 0 en 0 mm a 1 en Q75₇₂.
- Por debajo de Q90 las curvas coinciden, así que P75 sigue siendo la alerta temprana.
- En Q95 la escalada lleva H_P de 0.5 a 0.8, la frontera de la clase SIAT más alta.
- En cada píxel se usan sus propios percentiles; las medianas solo ilustran.

## 9. El viento: v0.1 contra v0.2

In [ ]:
r = np.linspace(0, 700, 701)
v01 = np.where(r <= 200, compute_wind_hazard_from_profile(r, vmax_kt=90.0, rmw_km=30.0), 0.0)
v02 = compute_w06_wind_hazard_with_tail(r, 200.0, 650.0, vmax_kt=90.0, lat=20.0, rmw_km=30.0, r34_mean_km=200.0, h34=0.2)
fig, ax = plt.subplots(figsize=(7, 4))
ax.axvspan(200, 650, color="#FCEBDD", lw=0, label="región indirecta")
ax.plot(r, v01, "--", color=COL["base"], lw=2, label="v0.1: Rankine, cero fuera de R34")
ax.plot(r, v02["H_W"], color=COL["v0.2"], lw=2.5, label="v0.2: W06 con cola (h34 = 0.2)")
ax.set(xlabel="distancia al centro (km)", ylabel="H_W", ylim=(0, 1.05), xlim=(0, 700),
       title="Huracán de 90 kt, 20°N, RMW 30 km, R34 200 km, R_W06 650 km")
ax.legend(frameon=False, fontsize=9)
fig.savefig(FIG / "perfil_viento_v01_v02.png", dpi=200, bbox_inches="tight")

**Cómo leer.**
- La v0.1 termina en R34: fuera de ahí, el viento no aporta.
- La v0.2 vale h34 en R34 y decae con el perfil W06 hasta cero en el radio de atribución, siendo continua en R34. La cola entra al componente indirecto: $H_{ind} = 1 - (1 - H_{W,ind})(1 - H_{P,ind})$.
- Detalle conocido: dentro del ojo la v0.2 vale h34 y no cero.

## 10. ¿Aporta información el viento fuera de R34?

Esta sección responde si el componente indirecto de viento está justificado. Para cada par se estima el viento sostenido W06 (PA21, sin calibrar a R34) en el municipio, a la distancia y con la intensidad del paso del pico de lluvia.

Hay que interpretarlo con cuidado: ese viento es también un indicador de cercanía e intensidad del ciclón. Correlaciona con marea de tensión, con lluvia fuera del píxel del pico y con exposición. Además, las declaratorias incluyen las de viento y marea.

In [ ]:
prof = Willoughby2006Profile(pares["vmax_pico_kt"].to_numpy(float) * KNOT_TO_MS, pares["lat_pmax"].to_numpy(float),
                             None, rmax_variant="pa21")
V = prof(pares["dist_pico_km"].to_numpy(float)) / KNOT_TO_MS                  # kt en el municipio
VMAX = pares["vmax_pico_kt"].to_numpy(float)
bandas = pd.cut(V, [0, 17, 24, 34, 50, 64, 999], right=False, labels=["<17", "17–24", "24–34", "34–50", "50–64", "≥64"])
pv = pares.assign(banda=bandas).groupby("banda", observed=False)["D"].agg(["size", "mean"])
print("AUC: lluvia (clase de P6) =", round(auc(pares["nivel_pico"].to_numpy(), D), 3),
      "| viento W06 en el municipio =", round(auc(V, D), 3))
pd.DataFrame({"n": pv["size"], "P(D) %": (100 * pv["mean"]).round(1)}).T

**Cómo leer.** La probabilidad de declaratoria crece con el viento estimado incluso por debajo de 34 kt, es decir, fuera de R34: de < 1 % con menos de 17 kt a varias veces la tasa base entre 17 y 34 kt. Fuera de R34 sí hay información. Falta decidir cómo representarla.

In [ ]:
# H_P aproximado desde el nivel de la v0.2 (valor medio de cada tramo de la rampa)
HP = np.array([0, 0.125, 0.375, 0.75, 1.0])[pares["nivel_v02"].fillna(0).astype(int).to_numpy()]


def h_viento(h34, v_out_kt):
    cola = h34 * np.clip((V - v_out_kt) / (34 - v_out_kt), 0, 1)
    dentro = h34 + (1 - h34) * np.clip((V - 34) / np.maximum(VMAX - 34, 1e-6), 0, 1)
    return np.where((V >= 34) & (VMAX > 34), dentro, cola)


def clase_siat(x):
    return np.digitize(x, [1e-9, 0.2 + 1e-9, 0.4 + 1e-9, 0.6 + 1e-9, 0.8 + 1e-9])


variantes = {"solo lluvia": None, "cola hasta R_W06 (2 m/s), h34 0.2 [v0.2 actual]": (0.2, 2.0 / KNOT_TO_MS),
             "cola hasta 17 kt, h34 0.2": (0.2, 17.0), "cola hasta 17 kt, h34 0.3": (0.3, 17.0),
             "cola hasta 24 kt, h34 0.2": (0.2, 24.0)}
filas = []
for nombre, par in variantes.items():
    hw = 0.0 if par is None else h_viento(*par)
    itchi = 1 - (1 - hw) * (1 - HP)
    cl = clase_siat(itchi)
    r = {"variante": nombre, "AUC": auc(itchi, D), "% pares con viento > 0": 100 * float(np.mean(np.asarray(hw) > 0))}
    for k in range(1, 6):
        r[f"PSS clase ≥ {k}"] = contingency(cl >= k, D)["PSS"]
    r["POFD clase ≥ 1"] = contingency(cl >= 1, D)["POFD"]
    filas.append(r)
VIENTO = pd.DataFrame(filas).set_index("variante")
VIENTO.to_csv(FIG / "viento_indirecto_variantes.csv", float_format="%.3f")
VIENTO.round(3)

**Cómo leer.** Aquí se combinan lluvia y viento como en ITCHI ($1-(1-H_W)(1-H_P)$) y se clasifican en clases SIAT de 0.2.
- **Cola hasta R_W06 (la v0.2 actual).** El AUC sube, porque el viento aporta, pero casi todos los pares (~96 %) quedan con viento > 0. Toda la zona de influencia pasa a clase 1 y la clase "muy bajo" deja de discriminar: POFD ≈ 0.97, PSS ≈ 0.03. Ese es el problema: con un radio de atribución de ~740 km, el perfil a 2 m/s es prácticamente brisa.
- **Cola hasta 17 kt.** Conserva casi toda la ganancia de AUC, solo ~10 % de los pares recibe viento y mejora el PSS en todas las clases, la 1 incluida.
- **Cola hasta 24 kt.** Es más conservadora. Tiene justificación física: con un factor de ráfaga de ~1.4, 24 kt sostenidos dan ráfagas de ~34 kt. Gana menos.

**Conclusión.** El componente indirecto de viento está justificado por los datos, pero la cola debe terminar donde el viento deja de ser relevante (~17–24 kt), no en el radio de atribución. Las aproximaciones de esta sección (H_P por tramos, W06 sin calibrar, viento en el paso del pico) bastan para comparar variantes, pero el valor final se debe calibrar con ITCHI completo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
for (nombre, row), c, ls in zip(VIENTO.iterrows(), ["#6B7280", "#D9822B", "#2A6FC9", "#14213D", "#8AB4E8"], ["--", ":", "-", "-", "-."]):
    ax.plot(range(1, 6), row[[f"PSS clase ≥ {k}" for k in range(1, 6)]], ls, marker="o", color=c, lw=2, label=nombre)
ax.set(xticks=range(1, 6), xlabel="alerta si clase SIAT ≥", ylabel="PSS", ylim=(0, 0.65))
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "viento_indirecto_pss.png", dpi=200, bbox_inches="tight")
print("figuras y tablas en", FIG)